# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [69]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


### 텍스트 데이터

In [70]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
TEXTS_BY_PAGE = loader.load()

In [71]:
for i, page in enumerate(TEXTS_BY_PAGE):
    page.metadata = {"page": i}

In [72]:
TEXTS_BY_PAGE[2].metadata

{'page': 2}

### 표 데이터

In [73]:
from img2table.document import PDF
from img2table.ocr import TesseractOCR

pdf = PDF(
    PDF_PATH, 
    detect_rotation=False,
    pdf_text_extraction=True
)

ocr = TesseractOCR(n_threads=1, lang="eng")

TABLES_BY_PAGE = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=30
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [74]:
# 표 데이터 - 메타데이터 title 보정

for page_num, tables in TABLES_BY_PAGE.items():
    for table in tables:
        if table and table.title:
            if len(table.title) > 20:
                table.title = None

In [75]:
import pandas as pd
def trim_table(df: pd.DataFrame):
    df.columns = df.iloc[0]
    df = df[1:]
    df.reset_index(drop=True, inplace=True)

    return df

In [97]:
# 표 데이터 metadata에 merge

to_delete_list = list()

for page_num, tables in TABLES_BY_PAGE.items():
    if tables:
        TEXTS_BY_PAGE[page_num].metadata["table"] = {
            f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
        }
    else:
        TEXTS_BY_PAGE[page_num].metadata["table"] = None
        to_delete_list.append(page_num)

for page_num in to_delete_list:
    del TABLES_BY_PAGE[page_num]

In [96]:
TEXTS_BY_PAGE[3].metadata["table"]["3.0"]

,직계존속,직계비속,형제자매,위탁아동
0,60세 이상\n(’64.12.31.이전),20세 이하\n(’04.01.01.이후),20세 이하\n60세 이상,해당과세기간 6개월 이상\n직접 양육한 위탁아동


In [ ]:
import pandas as pd
a = TABLES_BY_PAGE[3][0].df
a.columns = a.iloc[0]
a = a[1:]

a

b = pd.DataFrame(TABLES_BY_PAGE[3][0].df[1:], columns=TABLES_BY_PAGE[3][0].df.iloc[0])

In [ ]:
count = 0

for page_num, tables in TABLES_BY_PAGE.items():
    for idx, table in enumerate(tables):

        df = table.df

        print(f"Page {page_num} - Table {idx} DataFrame: ")
        display(df)
        count += 1

Page 2 - Table 0 DataFrame: 


,0,1,2,3
0,이용자,서비스,내 용,접근 경로
1,공통,원천징수(연말정산)안내 홈페이지,원천징수(연말정산)안내 홈페이지,국세청 홈페이지(www.nts.go.kr) \n국세신고안내  개인 또는 법인 ...
2,공통,연말정산 관련 질의회신 및 판례 조회,연말정산 관련 질의회신 및 판례 조회,국세법령정보시스템\n(www.hometax.go.kr  법령정보)
3,공통,인터넷 상담 및 회신,인터넷 상담 및 회신,국세청 국세상담센터\n(www.hometax.go.kr  상담/제보)
4,근로자,소득공제\n자료조회,연말정산 소득·세액공제 자료 조회,홈택스  장려금·연말정산·기부금 \n연말정산간소화\n[ 안내 ] 126-내선1-5번
5,근로자,소득공제\n자료조회,현금영수증 발행금액 조회,현금영수증\n(홈택스  전자(세금)계산서·현금영수증·\n신용카드  현금영수증(근...
6,근로자,공제신고서\n작성,간소화자료 선택 후 신고서 자동 반영,홈택스  장려금·연말정산·기부금 \n편리한 연말정산\n[ 안내 ] 126-내선1-5번
7,근로자,간편제출,작성된 공제신고서 및 증명자료\n온라인 제출,홈택스  장려금·연말정산·기부금 \n편리한 연말정산\n[ 안내 ] 126-내선1-5번
8,근로자,신고결과\n조회,과거 원천징수 영수증(지급명세서)조회\n* ’19년~’24년 및 ’25년 중 제출한...,홈택스  My홈택스  연말정산 \n지급명세서 등 제출내역\n[ 안내 ] 126...
9,근로자,안내책자\n및 영상,「근로자를 위한 연말정산 안내」 책자\n「근로자를 위한 연말정산」 동영상\n「편리한...,국세청 홈페이지(www.nts.go.kr) \n국세신고안내  개인 또는 법인 ...


Page 3 - Table 0 DataFrame: 


,0,1,2,3
0,직계존속,직계비속,형제자매,위탁아동
1,60세 이상\n(’64.12.31.이전),20세 이하\n(’04.01.01.이후),20세 이하\n60세 이상,해당과세기간 6개월 이상\n직접 양육한 위탁아동


Page 3 - Table 1 DataFrame: 


...

,0,1,2,3
0,구분,근로소득,사업소득,연금소득
1,None,모든 근로자\n(일용근로자 제외),"보험모집인, 방문판매원,\n음료배달원",None


Page 423 - Table 1 DataFrame: 


,0,1,2,3
0,None,총급여,사업소득 수입금액,None
1,3. 소득금액\n(①),근로소득금액\n(총급여 - 근로소득공제),사업소득금액\n(수입금액 × 소득률),연금소득금액\n(총연금액 - 연금소득공제)
2,4. 종합소득\n공제금액\n(②),"[종합소득공제]\n(기본공제, 추가공제,\n연금보험료공제,\n특별소득공제,\n그 밖...","[종합소득공제]\n(기본공제, 추가공제,\n연금보험료공제)","[종합소득공제]\n(기본공제, 추가공제,"
3,5. 과세표준\n(①-②),근로소득금액 -\n종합소득공제금액,사업소득금액 -\n종합소득공제금액,연금소득금액 -\n종합소득공제금액
4,6. 세율,6~45%,6~45%,6~45%
5,None,과세표준 × 세율,과세표준 × 세율,None


Page 424 - Table 0 DataFrame: 


,0,1,2,3,4
0,항목,항목,구분,구분,비고
1,항목,항목,외국인\n거주자,비거주자,비고
2,None,None,국외원천\n소득포함,국내 원천소득,｢소득세법｣ 제3조에 따른 단기거주 외국인은 국외\n원천소득 중 국내에서 지급되거나...
3,None,None,○,○,None
4,인적공제,기본공제,○,본인만 공제,None
5,인적공제,추가공제,○,본인만 공제,None
6,None,None,○,○,본인이 납부하는 국민연금보험료에 한함
7,특별 소득공제,건강·고용보험료 등,○,×,None
8,특별 소득공제,주택자금,○,×,None
9,그 밖의 소득공제,개인연금저축\n소기업 등 공제부금\n투자조합출자\n신용카드 등 사용금액\n고용유지중...,○,×,None


In [ ]:
TABLES_BY_PAGE[7][1].df

,0,1,2,3,4,5
0,구 분,구 분,기본공제대상자의 요건,기본공제대상자의 요건,근로기간 지출한\n비용만 공제,비 고
1,구 분,구 분,나이요건 소득요건,나이요건 소득요건,근로기간 지출한\n비용만 공제,비 고
2,특별 소득공제,보 험 료,None,None,None,None
3,특별 소득공제,주택자금공제,None,None,None,None
4,그 밖의 소득공제,개인연금저축,None,None,None,None
5,그 밖의 소득공제,주택마련저축,None,None,None,None
6,그 밖의 소득공제,신용카드 등,×,,,None
7,자녀세액공제 (8세이상),자녀세액공제 (8세이상),,,-,기본공제대상 자녀\n(입양자·위탁아동·손자녀 포함)


pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

큰 제목

내용 1

내용 2

    - 하위 내용 1

    - 하위 내용 2

      - 표 1

...

In [ ]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.